In [ ]:
import h5py
import numpy as np

with h5py.File("tree.hdf5", "r") as f:
    print(list(f.keys()))
SNAP = 85
with h5py.File("tree.hdf5", "r") as f:

    snap = f["SnapNum"][:]
    subfind = f["SubfindID"][:]

    matches = np.where(snap == 85)[0]

    print("Number of snapshot-66 entries:", len(matches))

    for i in matches:
        print(
            "Snap:", snap[i],
            "SubfindID:", subfind[i]
        )


In [ ]:
import requests

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"
headers = {"api-key": API_KEY}
SUBFIND_ID = 682178

url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/{SNAP}/subhalos/{SUBFIND_ID}/info.json"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

d = r.json()

print("Position:", d["SubhaloPos"])
print("Group:", d["SubhaloGrNr"])
print("Parent:", d.get("SubhaloParent"))
print("Mass:", d["SubhaloMass"])
print("HalfmassRad:", d["SubhaloHalfmassRad"])

In [ ]:
GROUP_ID = d["SubhaloGrNr"]

url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/{SNAP}/halos/{GROUP_ID}/info.json"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

group = r.json()

print("Group position:", group["GroupPos"])
print("R200:", group["Group_R_Crit200"])
print("M200:", group["Group_M_Crit200"])

In [ ]:
import numpy as np

galaxy_pos = np.array(d["SubhaloPos"])
group_pos = np.array(group["GroupPos"])

BOX_SIZE = 205000.0 
delta = galaxy_pos - group_pos
delta = delta - BOX_SIZE * np.round(delta / BOX_SIZE)

distance = np.linalg.norm(delta)

R200 = group["Group_R_Crit200"]

r_over_R200 = distance / R200

print("Galaxy-group distance:", distance, "ckpc/h")
print("R200:", R200, "ckpc/h")
print("r/R200:", r_over_R200)

In [ ]:
url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/66/subhalos/{SUBFIND_ID}/sublink/simple.json"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

data = r.json()

print("\nMAIN PROGENITOR:")
for item in data.get("Main", []):
    print(item)

print("\nMERGERS:")
for item in data.get("Mergers", []):
    print(item)

In [ ]:
import requests
import numpy as np
import pandas as pd

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"
headers = {"api-key": API_KEY}

Z0_ID = 682178

# Get main progenitor branch
url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/99/subhalos/{Z0_ID}/sublink/simple.json"
)

tree = requests.get(url, headers=headers).json()

main = {
    snap: subfind
    for snap, subfind in tree["Main"]
    if 85 <= snap <= 99
}

results = []

for snap in sorted(main):

    subfind = main[snap]
    sub_url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind}/info.json"
    )

    sub = requests.get(sub_url, headers=headers).json()

    galaxy_pos = np.array(sub["SubhaloPos"])
    group_id = sub["SubhaloGrNr"]
    group_url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/halos/{group_id}/info.json"
    )

    group = requests.get(group_url, headers=headers).json()

    group_pos = np.array(group["GroupPos"])
    group_first_sub = group["GroupFirstSub"]
    delta = galaxy_pos - group_pos
    BOX_SIZE = 205000.0
    delta -= BOX_SIZE * np.round(delta / BOX_SIZE)

    distance = np.linalg.norm(delta)

    results.append({
        "snap": snap,
        "subfind": subfind,
        "group": group_id,
        "GroupFirstSub": group_first_sub,
        "is_central": subfind == group_first_sub,
        "galaxy_pos": galaxy_pos,
        "group_pos": group_pos,
        "dx": delta[0],
        "dy": delta[1],
        "dz": delta[2],
        "distance": distance,
        "R200": group["Group_R_Crit200"],
        "r_R200": distance / group["Group_R_Crit200"]
    })

df = pd.DataFrame(results)

print(df[
    [
        "snap",
        "subfind",
        "group",
        "GroupFirstSub",
        "is_central",
        "distance",
        "R200",
        "r_R200"
    ]
].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
plot_data = tree.dropna(subset=["Redshift", "f_DM"]).copy()
plot_data = plot_data.sort_values("Redshift", ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    plot_data["Redshift"],
    plot_data["f_DM"],
    marker="o",
    markersize=3,
    linewidth=1.5,
    label=r"$f_{\rm DM}(<2R_h)$"
)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1.2,
    label=r"DMDG threshold: $f_{\rm DM}=0.5$"
)
ax.invert_xaxis()
ax.set_xlabel("Redshift")
ax.set_ylabel(r"$f_{\rm DM}(<2R_h)$")
ax.set_title("Evolution of Dark-Matter Fraction for Subhalo 355308")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(
    df["snap"],
    df["r_R200"],
    marker="o",
    linewidth=2
)

plt.xlabel("Snapshot")
plt.ylabel(r"$r/R_{200}$")
plt.title("Distance from Host Center at Snapshot 91 (Galaxy 738558)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()